# Evidencia de la capa Gold: tópico × año

Este notebook es la evidencia de la tabla Gold `topic_year`, que alimenta el modelo de tópicos emergentes,
y de los indicadores (KPI) que se reportan a partir de ella. Es de solo lectura: lee `data/gold/` (y nada
más) y no escribe en `data/`. La definición operacional y el diccionario de variables están en
[`documentacion/4.gold.md`](../documentacion/4.gold.md).

El corpus se congeló el 2026-10-09; los valores de este notebook describen ese corte.

**Cómo leer las celdas.** Antes de cada celda de código hay una celda que dice qué hace, qué lee, qué muestra
y cómo interpretar el resultado. Las celdas con `assert` son comprobaciones: si el dato publicado cambia,
fallan en lugar de mostrar un número distinto sin aviso. Después de cada resultado relevante hay una
**lectura** con los valores del corte.

**Cómo ejecutarlo.** En el contenedor, con el comando de
[`documentacion/5.operacion.md`](../documentacion/5.operacion.md). Fuera del contenedor, desde el repositorio
con Python 3.11 y `requirements.txt`; la primera celda detecta dónde está y usa una sesión de Spark liviana.

## Contenido

0. Preparación: rutas, sesión de Spark y tablas en memoria.
1. Manifiesto de Gold.
2. Filas, filas etiquetables y positivos.
3. Distribución de clases por año base.
4. Sensibilidad de la etiqueta: umbral de crecimiento × soporte mínimo.
5. Tamaños y positivos por partición temporal.
6. Variables: tasa de nulos y medias por clase en filas etiquetables.
7. Los 20 tópico-años emergentes con mayor crecimiento, para revisión manual.
8. Indicadores (KPI): crecimiento, concentración, perfil de la producción, emergencia, subcampos, señales
   actuales y tabla resumen.

## 0. Preparación

**Qué hace.** Ubica el proyecto y los datos, abre una sesión de Spark y deja en memoria dos tablas que usan
todas las secciones siguientes.

- Dentro del contenedor usa `/app` (código) y `/data` (datos), con `local[*]` y 4 GB para Spark.
- Fuera del contenedor busca hacia arriba la carpeta que tiene `src/` y `pyproject.toml`, lee `data/` de ahí y
  usa `local[2]` y 2 GB. Las variables `EVIDENCE_SPARK_MASTER` y `EVIDENCE_SPARK_DRIVER_MEMORY` cambian esos
  valores.

**Lee.** `data/gold/topic_year`: una fila por tópico y año (2011–2025), con las variables y la etiqueta.

**Deja en memoria.**
- `topic_year`: la tabla completa.
- `labelable`: solo las filas con etiqueta (`target_emerging` no nulo), que son las que usa el modelo.

**Muestra.** Una línea con las rutas y la configuración de Spark elegidas, para saber de dónde salieron los
datos. Los avisos de Spark al iniciar (`WARN`) son normales.

In [1]:
from pathlib import Path
import json
import os
import sys

from pyspark.sql import SparkSession, functions as F

# Container paths win; outside Docker, fall back to the repository checkout.
IN_CONTAINER = (Path("/app") / "src").is_dir()
if IN_CONTAINER:
    PROJECT_ROOT, DATA = Path("/app"), Path("/data")
else:
    PROJECT_ROOT = next(
        (p for p in (Path.cwd(), *Path.cwd().parents) if (p / "src").is_dir() and (p / "pyproject.toml").is_file()),
        None,
    )
    if PROJECT_ROOT is None:
        raise RuntimeError(f"Repository root not found from {Path.cwd()}; open the notebook from inside the project.")
    DATA = PROJECT_ROOT / "data"
# Inside the container keep the original Spark sizing; on a laptop default to a light session.
SPARK_MASTER = os.environ.get("EVIDENCE_SPARK_MASTER", "local[*]" if IN_CONTAINER else "local[2]")
SPARK_DRIVER_MEMORY = os.environ.get("EVIDENCE_SPARK_DRIVER_MEMORY", "4g" if IN_CONTAINER else "2g")
print(f"project={PROJECT_ROOT} data={DATA} spark={SPARK_MASTER} driver_memory={SPARK_DRIVER_MEMORY}")
GOLD = DATA / "gold"
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.gold_features import FEATURE_COLUMNS
from src.gold_rules import HORIZON_YEARS, LAST_COMPLETE_YEAR, MIN_SUPPORT, emergence_label_column

spark = (
    SparkSession.builder.master(SPARK_MASTER)
    .appName("gold-evidence")
    .config("spark.ui.enabled", "false")
    .config("spark.ui.showConsoleProgress", "false")
    .config("spark.driver.memory", SPARK_DRIVER_MEMORY)
    .getOrCreate()
)
spark.sparkContext.setLogLevel("ERROR")

# topic_year: every topic x year row. labelable: rows with a label, the ones the model trains and is evaluated on.
topic_year = spark.read.parquet(str(GOLD / "topic_year")).cache()
labelable = topic_year.filter(F.col("target_emerging").isNotNull()).cache()

project=/app data=/data spark=local[*] driver_memory=4g


Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/10/10 23:59:30 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


## 1. Manifiesto

**Qué hace.** Lee `_manifest.json`, el archivo que la publicación de Gold escribe al final y que describe
exactamente las tablas publicadas.

**Muestra.** Versión de Gold y de Silver, regla de la etiqueta (umbral de crecimiento, horizonte, soporte
mínimo, último año completo), conteos (filas, etiquetables, positivos, tópicos, publicaciones del universo) y
ventana de descarga de OpenAlex (instantánea de la taxonomía).

**Comprobación.** La lista de variables del manifiesto debe ser idéntica a `FEATURE_COLUMNS` del código; si
no, la tabla publicada no corresponde al código actual.

In [2]:
# 1. Manifest: written last by the Gold publish; describes the published tables.
# Prints every manifest field except the long feature list, then checks that list against the code.
manifest = json.loads((GOLD / "_manifest.json").read_text(encoding="utf-8"))
print(json.dumps({key: value for key, value in manifest.items() if key != "feature_columns"}, indent=2, sort_keys=True))
print(f"feature_columns: {len(manifest['feature_columns'])}")
assert manifest["feature_columns"] == list(FEATURE_COLUMNS)

{
  "created_at": "2026-10-10T12:59:24.838923+00:00",
  "first_year": 2011,
  "gold_version": 2,
  "labeled_rows": 3308,
  "last_year": 2025,
  "positive_rows": 289,
  "rows": 52380,
  "rule": {
    "growth_threshold": 0.5,
    "horizon_years": 2,
    "last_complete_year": 2025,
    "min_support": 10
  },
  "silver_schema_version": 4,
  "source": "openalex",
  "taxonomy_snapshot": {
    "ingested_at_max": "2026-10-09T05:34:07.761256+00:00",
    "ingested_at_min": "2026-10-03T01:26:02.861386+00:00"
  },
  "topic_assignment": "primary_topic",
  "topic_count": 3492,
  "universe_rows": 300171
}
feature_columns: 24


## 2. Filas, etiquetables y positivos

**Qué hace.** Cuenta la tabla publicada y comprueba sus criterios de aceptación.

**Muestra.**
- `rows`: filas totales; deben ser 3 492 tópicos × 15 años = 52 380.
- `topics` y `unique_topic_years`: tópicos distintos y pares (`topic_id`, `year`) distintos; si los pares
  coinciden con las filas, no hay duplicados.
- `labelable`: filas con etiqueta (3 308). `positives`: filas emergentes (289). `positive_rate`: su cociente.

**Comprobaciones.** Cada fila etiquetada debe cerrar su horizonte en `year + 2`, y ninguna fila con menos de
10 publicaciones o con año base posterior a 2023 puede tener etiqueta. Ambos conteos deben ser 0.

In [3]:
# 2. Acceptance counts: rows, duplicates, labelable rows and positives, then two label consistency checks.
counts = topic_year.agg(
    F.count(F.lit(1)).alias("rows"),
    F.countDistinct("topic_id").alias("topics"),
    F.countDistinct("topic_id", "year").alias("unique_topic_years"),
    F.count("target_emerging").alias("labelable"),
    F.sum("target_emerging").alias("positives"),
).withColumn("positive_rate", F.round(F.col("positives") / F.col("labelable"), 4))
counts.show()
row = counts.first()
assert (row["rows"], row["unique_topic_years"], row["labelable"], row["positives"]) == (52380, 52380, 3308, 289)
bad_end = labelable.filter(F.col("label_end_year") != F.col("year") + HORIZON_YEARS).count()
bad_null = topic_year.filter(((F.col("n_t") < 10) | (F.col("year") > 2023)) & F.col("target_emerging").isNotNull()).count()
print(f"label_end_year != year + {HORIZON_YEARS}: {bad_end}; labeled rows with n_t < 10 or year > 2023: {bad_null}")

+-----+------+------------------+---------+---------+-------------+
| rows|topics|unique_topic_years|labelable|positives|positive_rate|
+-----+------+------------------+---------+---------+-------------+
|52380|  3492|             52380|     3308|      289|       0.0874|
+-----+------+------------------+---------+---------+-------------+



label_end_year != year + 2: 0; labeled rows with n_t < 10 or year > 2023: 0


## 3. Distribución de clases por año base

**Qué hace.** Resume la tabla por año base.

**Muestra.** Por año: filas (siempre 3 492, la grilla es densa), `observed_topics` (tópicos con al menos una
publicación), filas etiquetables, positivos y tasa de positivos.

**Cómo leerlo.** Los años 2024 y 2025 no tienen etiqueta porque su horizonte termina después de 2025, el último
año completo. La tasa de positivos por año es la base de la sección 8.4.

In [4]:
# 3. Class distribution per base year: observed topics, labelable rows, positives and positive rate.
(
    topic_year.groupBy("year")
    .agg(
        F.count(F.lit(1)).alias("rows"),
        F.sum(F.when(F.col("n_t") > 0, 1).otherwise(0)).alias("observed_topics"),
        F.count("target_emerging").alias("labelable"),
        F.sum("target_emerging").alias("positives"),
    )
    .withColumn("positive_rate", F.round(F.col("positives") / F.col("labelable"), 4))
    .orderBy("year")
    .show(20)
)

+----+----+---------------+---------+---------+-------------+
|year|rows|observed_topics|labelable|positives|positive_rate|
+----+----+---------------+---------+---------+-------------+
|2011|3492|            976|      113|        3|       0.0265|
|2012|3492|            982|      121|       10|       0.0826|
|2013|3492|           1149|      150|       17|       0.1133|
|2014|3492|           1079|      151|       19|       0.1258|
|2015|3492|           1069|      134|       17|       0.1269|
|2016|3492|           1084|      149|       18|       0.1208|
|2017|3492|           1209|      185|       19|       0.1027|
|2018|3492|           1377|      256|       20|       0.0781|
|2019|3492|           1518|      322|       23|       0.0714|
|2020|3492|           1733|      365|       19|       0.0521|
|2021|3492|           1917|      416|       28|       0.0673|
|2022|3492|           2015|      452|       51|       0.1128|
|2023|3492|           2153|      494|       45|       0.0911|
|2024|34

## 4. Sensibilidad de la etiqueta

**Qué hace.** Vuelve a aplicar la misma función Spark de la regla (`emergence_label_column` de
`src/gold_rules.py`) sobre las columnas de prevalencia de `topic_year`, cambiando solo el umbral de
crecimiento (0,30; 0,50; 0,70) y el soporte mínimo (5, 10, 20). No hay una segunda implementación de la
regla.

**Muestra.** Para cada combinación: filas etiquetables, positivos y tasa de positivos.

**Comprobación.** La fila con `growth = 0.5` y `min_support = 10` (los parámetros publicados) debe reproducir
3 308 etiquetables y 289 positivos.

In [5]:
# 4. Sensitivity grid: re-apply the same Spark label builder with each threshold and minimum support.
path = [F.col("prevalence_t"), *(F.col(f"label_prevalence_t{step}") for step in range(1, HORIZON_YEARS + 1))]
grid_rows = []
for growth_threshold in (0.30, 0.50, 0.70):
    for min_support in (5, 10, 20):
        label = emergence_label_column(
            F.col("n_t"), path, F.col("year"), growth_threshold=growth_threshold, min_support=min_support
        )
        stats = topic_year.select(label.alias("label")).agg(
            F.count("label").alias("labelable"), F.sum("label").alias("positives")
        ).first()
        grid_rows.append((growth_threshold, min_support, stats["labelable"], int(stats["positives"] or 0)))
sensitivity = spark.createDataFrame(
    grid_rows, "growth_threshold double, min_support int, labelable long, positives long"
).withColumn("positive_rate", F.round(F.col("positives") / F.col("labelable"), 4))
sensitivity.orderBy("growth_threshold", "min_support").show()
default = sensitivity.filter((F.col("growth_threshold") == 0.5) & (F.col("min_support") == 10)).first()
assert (default["labelable"], default["positives"]) == (3308, 289)

+----------------+-----------+---------+---------+-------------+
|growth_threshold|min_support|labelable|positives|positive_rate|
+----------------+-----------+---------+---------+-------------+
|             0.3|          5|     5867|      645|       0.1099|
|             0.3|         10|     3308|      386|       0.1167|
|             0.3|         20|     1746|      196|       0.1123|
|             0.5|          5|     5867|      511|       0.0871|
|             0.5|         10|     3308|      289|       0.0874|
|             0.5|         20|     1746|      145|        0.083|
|             0.7|          5|     5867|      393|        0.067|
|             0.7|         10|     3308|      214|       0.0647|
|             0.7|         20|     1746|      100|       0.0573|
+----------------+-----------+---------+---------+-------------+



**Lectura (corte 2026-10-09).** El soporte mínimo cambia cuántas filas son etiquetables (5 867, 3 308 y
1 746 para 5, 10 y 20), pero casi no mueve la tasa de positivos con un umbral dado. El umbral de
crecimiento sí la mueve: alrededor de 11 % con 0,30, 8,7 % con 0,50 y 6 % con 0,70. Los parámetros por
defecto quedan en el centro de la grilla.

## 5. Particiones temporales

Una fila de entrenamiento es válida para una ventana de evaluación que empieza en el año E cuando
`label_end_year <= E`: su etiqueta ya se conoce al final de E, cuando se hace la predicción.

| Partición | Años base | Regla |
|---|---|---|
| Entrenamiento para validación | 2011–2017 | `label_end_year <= 2019` |
| Validación | 2019–2020 | evaluación |
| Reentrenamiento final | 2011–2020 | `label_end_year <= 2022` |
| Prueba final | 2022–2023 | evaluación |

Los años base 2018 y 2021 no entran en el entrenamiento de su ventana porque su etiqueta termina
después de E.

**Muestra.** Para cada partición: primer y último año base, filas, positivos y tasa de positivos.

In [6]:
# 5. Temporal splits: size and positives of each split under the rule label_end_year <= E.
splits = [
    ("train_for_validation", F.col("label_end_year") <= 2019),
    ("validation", F.col("year").between(2019, 2020)),
    ("retrain_final", F.col("label_end_year") <= 2022),
    ("test", F.col("year").between(2022, 2023)),
]
split_rows = []
for name, condition in splits:
    part = labelable.filter(condition)
    stats = part.agg(
        F.count(F.lit(1)).alias("rows"), F.sum("target_emerging").alias("positives"),
        F.min("year").alias("first_base_year"), F.max("year").alias("last_base_year"),
    ).first()
    split_rows.append((name, stats["first_base_year"], stats["last_base_year"], stats["rows"], int(stats["positives"])))
(
    spark.createDataFrame(split_rows, "split string, first_base_year int, last_base_year int, rows long, positives long")
    .withColumn("positive_rate", F.round(F.col("positives") / F.col("rows"), 4))
    .show(truncate=False)
)

+--------------------+---------------+--------------+----+---------+-------------+
|split               |first_base_year|last_base_year|rows|positives|positive_rate|
+--------------------+---------------+--------------+----+---------+-------------+
|train_for_validation|2011           |2017          |1003|103      |0.1027       |
|validation          |2019           |2020          |687 |42       |0.0611       |
|retrain_final       |2011           |2020          |1946|165      |0.0848       |
|test                |2022           |2023          |946 |96       |0.1015       |
+--------------------+---------------+--------------+----+---------+-------------+



**Lectura.** La validación tiene una tasa de positivos menor (6,1 %) que su entrenamiento (10,3 %), y la
prueba final una mayor (10,2 %) que su reentrenamiento (8,5 %). Con 42 y 96 positivos, las métricas de
evaluación tendrán intervalos amplios; conviene informar métricas por año base y no solo agregadas.

## 6. Variables en filas etiquetables

**Qué hace.** Para cada variable numérica de `FEATURE_COLUMNS`, calcula sobre las filas etiquetables la tasa
de nulos y la media en cada clase.

**Muestra.**
- `null_rate`: proporción de filas sin valor. Los nulos son intencionales (rezagos antes de 2011, cocientes con
  denominador cero o nulo); Gold no imputa.
- `mean_class_0` y `mean_class_1`: media en filas no emergentes y emergentes. Una diferencia grande entre ambas
  indica una variable con señal para el modelo.
- `age_left_censored` se muestra como proporción. `subfield`, `field` y `domain` son categóricas y se resumen
  aparte por su número de valores distintos.

In [7]:
# 6. Feature profile on labelable rows: null rate and mean per class (Gold does not impute).
categorical = ("subfield", "field", "domain")
numeric = [name for name in FEATURE_COLUMNS if name not in categorical]
aggregates = []
for name in numeric:
    value = F.col(name).cast("double")
    aggregates += [
        F.avg(F.when(value.isNull(), 1.0).otherwise(0.0)).alias(f"{name}__null"),
        F.avg(F.when(F.col("target_emerging") == 0, value)).alias(f"{name}__mean0"),
        F.avg(F.when(F.col("target_emerging") == 1, value)).alias(f"{name}__mean1"),
    ]
stats = labelable.agg(*aggregates).first()
feature_rows = [
    (name, float(stats[f"{name}__null"]), stats[f"{name}__mean0"], stats[f"{name}__mean1"]) for name in numeric
]
(
    spark.createDataFrame(feature_rows, "feature string, null_rate double, mean_class_0 double, mean_class_1 double")
    .select("feature", *(F.round(name, 4).alias(name) for name in ("null_rate", "mean_class_0", "mean_class_1")))
    .show(len(feature_rows), truncate=False)
)
labelable.agg(*(F.countDistinct(name).alias(f"distinct_{name}") for name in categorical)).show()

+--------------------------+---------+------------+------------+
|feature                   |null_rate|mean_class_0|mean_class_1|
+--------------------------+---------+------------+------------+
|n_t                       |0.0      |43.2849     |40.9758     |
|prevalence_t              |0.0      |0.0029      |0.0026      |
|prevalence_percentile_t   |0.0      |0.9366      |0.9355      |
|prevalence_t_minus_1      |0.0342   |0.0028      |0.0022      |
|prevalence_t_minus_2      |0.0707   |0.0027      |0.002       |
|growth_1y                 |0.0354   |0.2473      |0.2749      |
|growth_2y_past            |0.0789   |0.4017      |0.5497      |
|acceleration              |0.0795   |0.0454      |-0.0155     |
|consecutive_growth_years  |0.0342   |0.9952      |1.2552      |
|years_since_first_observed|0.0      |7.5071      |7.3287      |
|age_left_censored         |0.0      |0.8632      |0.7924      |
|unique_authors_t          |0.0      |149.3647    |125.9619    |
|unique_institutions_t   

+-----------------+--------------+---------------+
|distinct_subfield|distinct_field|distinct_domain|
+-----------------+--------------+---------------+
|              138|            24|              4|
+-----------------+--------------+---------------+



## 7. Tópico-años emergentes para revisión manual

**Qué hace.** Lista los 20 positivos con mayor crecimiento de prevalencia en el horizonte
(`label_growth_2y`, columna de auditoría que nunca es variable del modelo).

**Muestra.** Nombre del tópico, subcampo, año base, publicaciones en ese año (`n_t`), prevalencia y
crecimiento en el horizonte (2,0 = la prevalencia se triplicó).

**Cómo leerlo.** Sirve para comprobar que la etiqueta marca temas plausibles y no artefactos del muestreo.

In [8]:
# 7. Manual review: the 20 emergent topic-years with the largest prevalence growth over the horizon.
(
    labelable.filter(F.col("target_emerging") == 1)
    .orderBy(F.desc(f"label_growth_{HORIZON_YEARS}y"), "topic_id", "year")
    .select(
        "topic_name", "subfield", "year", "n_t",
        F.round("prevalence_t", 6).alias("prevalence_t"),
        F.round(f"label_growth_{HORIZON_YEARS}y", 3).alias(f"label_growth_{HORIZON_YEARS}y"),
    )
    .show(20, truncate=60)
)

+---------------------------------------------------+---------------------------------------+----+---+------------+---------------+
|                                         topic_name|                               subfield|year|n_t|prevalence_t|label_growth_2y|
+---------------------------------------------------+---------------------------------------+----+---+------------+---------------+
|               Artificial Intelligence in Education|                    Information Systems|2022|139|    0.004981|          4.475|
|               Advanced Neural Network Applications|Computer Vision and Pattern Recognition|2014| 44|    0.006563|          4.323|
|                Multi-Agent Systems and Negotiation|                Artificial Intelligence|2023| 11|     3.21E-4|          4.004|
|                                     Topic Modeling|                Artificial Intelligence|2022|289|    0.010356|          3.588|
|                      Face recognition and analysis|Computer Vision and Pat

**Revisión manual.** La mayoría de los casos son temas plausibles de crecimiento real en el periodo
(IA en educación, redes neuronales profundas desde 2012–2014, robustez adversarial, IoT y edge
computing). Hay un artefacto visible: «COVID-19 diagnosis using AI» aparece como emergente con años base
2018 y 2019. El tópico es una asignación de OpenAlex hecha con la taxonomía actual, que agrupa
publicaciones previas (por ejemplo, diagnóstico de neumonía por imagen) bajo un nombre posterior. La
etiqueta mide crecimiento de prevalencia del identificador de tópico, no la novedad del nombre; ver las
limitaciones en [`documentacion/4.gold.md`](../documentacion/4.gold.md).

## 8. Indicadores (KPI)

Gold no publica una tabla de KPI: los indicadores se calculan aquí a partir de `topic_year` y `year_totals`,
con definiciones fijas, para que cada número reportado tenga su evidencia.

**Advertencia de muestreo.** Gold es una muestra: cada año contiene alrededor del 6,4 % de las publicaciones
planificadas (`sampling_fraction`, columna de `year_totals`). Los **conteos** son de la muestra y no deben
reportarse como totales del campo. Las **tasas, proporciones y crecimientos** sí son reportables, porque la
fracción muestreada es estable entre años (sección 8.1).

| Sección | KPI | Definición |
|---|---|---|
| 8.1 | Crecimiento | Variación interanual y crecimiento anual compuesto de las publicaciones; tópicos activos (`n_t > 0`). |
| 8.2 | Concentración | Participación de los 10 tópicos más grandes; índice de Herfindahl (HHI = Σ prevalencia²) y tópicos efectivos (1 / HHI). |
| 8.3 | Perfil de la producción | Acceso abierto, conferencias, revisiones, autores por publicación y autores nuevos, ponderados por publicaciones. |
| 8.4 | Emergencia | Tasa de positivos global y del último año base etiquetado; tópicos emergentes de ese año. |
| 8.5 | Subcampos | Participación de cada subcampo en las publicaciones del año, al inicio y al final del último horizonte de 2 años. |
| 8.6 | Señales actuales | Tópicos del último año completo con crecimiento reciente; todavía sin etiqueta. |
| 8.7 | Resumen | Una fila por KPI, con su periodo y valor. |

### 8.1 Crecimiento

**Qué hace.** Une `year_totals` con los totales por año de `topic_year` y calcula, por año, la variación
interanual de publicaciones y los tópicos activos. Después calcula el crecimiento anual compuesto entre el
primer y el último año.

**Lee.** `data/gold/year_totals`: por año, `n_total` (publicaciones de la muestra), `plan_total` (publicaciones
planificadas) y `sampling_fraction` (su cociente).

**Muestra.** `n_total`, `yoy_growth_pct` (variación respecto del año anterior, en %), `active_topics` y
`sampling_pct` (fracción muestreada, en %). Al final, una línea con el crecimiento anual compuesto.

**Comprobaciones.** Cada publicación cuenta una vez bajo su tópico principal, así que la suma de `n_t` de un
año debe ser igual a `n_total` y las prevalencias del año deben sumar 1.

In [9]:
# 8.1 Growth KPIs: sample publications per year, year-over-year change, active topics and compound growth.
from pyspark.sql import Window

year_totals = spark.read.parquet(str(GOLD / "year_totals"))
per_year = topic_year.groupBy("year").agg(
    F.sum("n_t").alias("sum_n_t"),
    F.sum("prevalence_t").alias("sum_prevalence"),
    F.sum((F.col("n_t") > 0).cast("int")).alias("active_topics"),
)
growth_kpis = (
    year_totals.join(per_year, "year")
    .withColumn("yoy_growth", F.col("n_total") / F.lag("n_total").over(Window.orderBy("year")) - 1)
    .orderBy("year")
)
# Each publication counts once under its primary topic: topic counts add up to the year total, prevalences to 1.
assert growth_kpis.filter(F.col("sum_n_t") != F.col("n_total")).count() == 0
assert growth_kpis.filter(F.abs(F.col("sum_prevalence") - 1) > 1e-9).count() == 0
growth_kpis.select(
    "year",
    "n_total",
    F.round(F.col("yoy_growth") * 100, 1).alias("yoy_growth_pct"),
    "active_topics",
    F.round(F.col("sampling_fraction") * 100, 2).alias("sampling_pct"),
).show(20)
growth_by_year = {row["year"]: row for row in growth_kpis.collect()}
first_year, last_year = min(growth_by_year), max(growth_by_year)
cagr = (growth_by_year[last_year]["n_total"] / growth_by_year[first_year]["n_total"]) ** (1 / (last_year - first_year)) - 1
print(f"compound annual growth {first_year}-{last_year}: {cagr:.1%}")

+----+-------+--------------+-------------+------------+
|year|n_total|yoy_growth_pct|active_topics|sampling_pct|
+----+-------+--------------+-------------+------------+
|2011|   5097|          NULL|          976|        6.31|
|2012|   5301|           4.0|          982|        6.28|
|2013|   6589|          24.3|         1149|        7.32|
|2014|   6704|           1.7|         1079|        7.14|
|2015|   6279|          -6.3|         1069|        6.18|
|2016|   6841|           9.0|         1084|        5.89|
|2017|   8881|          29.8|         1209|        6.17|
|2018|  12487|          40.6|         1377|        6.45|
|2019|  16235|          30.0|         1518|        6.36|
|2020|  20377|          25.5|         1733|        6.45|
|2021|  24227|          18.9|         1917|        6.44|
|2022|  27906|          15.2|         2015|        6.56|
|2023|  34222|          22.6|         2153|        6.26|
|2024|  46277|          35.2|         2269|        6.32|
|2025|  72748|          57.2|  

compound annual growth 2011-2025: 20.9%


**Lectura (corte 2026-10-09).** Las publicaciones crecen a un 20,9 % anual compuesto entre 2011 y 2025, y 2025
tiene la mayor variación interanual de la serie (+57,2 %). 2015 es el único año con caída (−6,3 %). Los
tópicos activos pasan de 976 a 2 727 (×2,8): el campo no solo crece, también se abre a más temas. La fracción
muestreada se mantiene entre 5,9 % y 7,3 % sin tendencia, por eso las tasas son comparables entre años.

### 8.2 Concentración temática

**Qué hace.** Ordena los tópicos de cada año por publicaciones y mide cuánto de la producción se concentra en
pocos tópicos.

**Muestra.**
- `top10_share_pct`: porcentaje de las publicaciones del año que pertenece a los 10 tópicos más grandes.
- `hhi_x10000`: índice de Herfindahl (suma de las prevalencias al cuadrado) × 10 000. Sube cuando la producción
  se concentra.
- `effective_topics`: 1 / HHI, el número de tópicos de igual tamaño que daría la misma concentración. Es la
  forma más intuitiva de leer el HHI.

In [10]:
# 8.2 Concentration KPIs per year: share of the 10 largest topics, Herfindahl index and effective number of topics.
by_size = Window.partitionBy("year").orderBy(F.desc("n_t"), "topic_id")
concentration = (
    topic_year.withColumn("size_rank", F.row_number().over(by_size))
    .groupBy("year")
    .agg(
        F.sum(F.when(F.col("size_rank") <= 10, F.col("prevalence_t"))).alias("top10_share"),
        F.sum(F.col("prevalence_t") * F.col("prevalence_t")).alias("hhi"),
    )
    .orderBy("year")
)
concentration.select(
    "year",
    F.round(F.col("top10_share") * 100, 1).alias("top10_share_pct"),
    F.round(F.col("hhi") * 10000, 1).alias("hhi_x10000"),
    F.round(1 / F.col("hhi"), 0).cast("int").alias("effective_topics"),
).show(20)
concentration_by_year = {row["year"]: row for row in concentration.collect()}

+----+---------------+----------+----------------+
|year|top10_share_pct|hhi_x10000|effective_topics|
+----+---------------+----------+----------------+
|2011|           23.4|     124.9|              80|
|2012|           22.4|      98.8|             101|
|2013|           20.3|      87.8|             114|
|2014|           21.2|      88.7|             113|
|2015|           20.4|      75.4|             133|
|2016|           20.6|      77.3|             129|
|2017|           17.8|      66.6|             150|
|2018|           18.2|      63.9|             156|
|2019|           17.9|      61.1|             164|
|2020|           18.8|      63.4|             158|
|2021|           19.0|      63.4|             158|
|2022|           19.4|      66.3|             151|
|2023|           23.8|     110.6|              90|
|2024|           26.9|     135.4|              74|
|2025|           26.3|     130.0|              77|
+----+---------------+----------+----------------+



**Lectura (corte 2026-10-09).** La producción se diversifica hasta 2019–2021 y se vuelve a concentrar desde
2023: el HHI pasa de 125 en 2011 a un mínimo de 61 en 2019 y sube a 130 en 2025. En tópicos efectivos, de 80
a 164 y de vuelta a 77. Los 10 tópicos más grandes concentran el 26,3 % de 2025, más que en 2011 (23,4 %). El
salto ocurre entre 2022 y 2023; este notebook no identifica qué tópicos lo explican.

### 8.3 Perfil de la producción

**Qué hace.** Calcula, por año, promedios de las variables de perfil de cada tópico ponderados por sus
publicaciones (`n_t`), de modo que un tópico grande pese en proporción a su tamaño. Usa solo filas con
publicaciones.

**Muestra.**
- `open_access`: proporción de publicaciones en acceso abierto.
- `conference`: proporción publicada en conferencias.
- `review`: proporción de artículos de revisión.
- `authors_per_pub`: autores promedio por publicación.
- `new_authors`: proporción de autores que no habían publicado antes en el tópico.

In [11]:
# 8.3 Production profile per year, weighted by publications (n_t) so each topic counts in proportion to its size.
def weighted_mean(column):
    weight = F.when(F.col(column).isNotNull(), F.col("n_t"))
    return F.sum(F.col("n_t") * F.col(column)) / F.sum(weight)


profile = (
    topic_year.filter(F.col("n_t") > 0)
    .groupBy("year")
    .agg(
        weighted_mean("oa_ratio_t").alias("open_access"),
        weighted_mean("conference_ratio_t").alias("conference"),
        weighted_mean("review_ratio_t").alias("review"),
        weighted_mean("authors_per_pub_t").alias("authors_per_pub"),
        weighted_mean("new_author_ratio_t").alias("new_authors"),
    )
    .orderBy("year")
)
profile.select("year", *(F.round(name, 3).alias(name) for name in profile.columns[1:])).show(20)
profile_by_year = {row["year"]: row for row in profile.collect()}

+----+-----------+----------+------+---------------+-----------+
|year|open_access|conference|review|authors_per_pub|new_authors|
+----+-----------+----------+------+---------------+-----------+
|2011|      0.229|     0.303| 0.002|          2.854|        1.0|
|2012|      0.279|     0.283| 0.003|          2.964|      0.935|
|2013|      0.305|     0.266| 0.003|          3.002|      0.921|
|2014|      0.324|     0.307| 0.004|          3.123|      0.893|
|2015|      0.344|     0.366| 0.004|          3.321|      0.886|
|2016|      0.359|     0.409| 0.001|          3.452|      0.895|
|2017|       0.41|     0.392| 0.004|          3.625|      0.904|
|2018|      0.444|     0.341| 0.006|          3.814|      0.915|
|2019|      0.491|     0.291|  0.01|          3.939|      0.912|
|2020|      0.538|     0.232| 0.012|          4.115|      0.909|
|2021|      0.559|     0.235|  0.02|           4.19|      0.904|
|2022|      0.583|     0.241| 0.021|          4.317|      0.904|
|2023|      0.632|      0

**Lectura (corte 2026-10-09).** El acceso abierto casi se triplica (22,9 % en 2011, 64,1 % en 2025). Las
conferencias alcanzan su máximo en 2016 (40,9 %) y bajan a 25,5 %: la producción se desplaza hacia revistas.
Los autores por publicación suben de 2,85 a 4,07 (máximo de 4,32 en 2022), es decir, más colaboración. Las
revisiones pasan de 0,2 % a 2,9 %. La proporción de autores nuevos se mantiene entre 0,89 y 0,94 desde 2012
(en 2011 vale 1,0 porque es el primer año observado: todos los autores son nuevos).

### 8.4 Emergencia

**Qué hace.** Resume la etiqueta, que es el resultado que el modelo debe anticipar: qué proporción de los
tópico-años con soporte suficiente resultan emergentes.

**Muestra.**
- Una línea con la tasa de emergencia global (positivos / etiquetables, 2011–2023) y la del último año base
  etiquetado (2023). La tasa por año está en la sección 3.
- Los tópicos emergentes de ese último año, ordenados por crecimiento en el horizonte.

In [12]:
# 8.4 Emergence KPIs: positive rate overall and in the last labelled base year, plus that year's emergent topics.
last_labelled_year = labelable.agg(F.max("year")).first()[0]


def emergence_rate(frame):
    return frame.agg(F.count(F.lit(1)).alias("labelable"), F.sum("target_emerging").alias("positives")).first()


overall = emergence_rate(labelable)
latest = emergence_rate(labelable.filter(F.col("year") == last_labelled_year))
print(
    f"emergence rate overall: {overall['positives'] / overall['labelable']:.1%} "
    f"({overall['positives']} / {overall['labelable']}); "
    f"base year {last_labelled_year}: {latest['positives'] / latest['labelable']:.1%} "
    f"({latest['positives']} / {latest['labelable']})"
)
(
    labelable.filter((F.col("year") == last_labelled_year) & (F.col("target_emerging") == 1))
    .orderBy(F.desc(f"label_growth_{HORIZON_YEARS}y"), "topic_id")
    .select(
        "topic_name", "subfield", "n_t",
        F.round(f"label_growth_{HORIZON_YEARS}y", 2).alias(f"label_growth_{HORIZON_YEARS}y"),
    )
    .show(15, truncate=60)
)

emergence rate overall: 8.7% (289 / 3308); base year 2023: 9.1% (45 / 494)
+--------------------------------------------------+-----------------------------+---+---------------+
|                                        topic_name|                     subfield|n_t|label_growth_2y|
+--------------------------------------------------+-----------------------------+---+---------------+
|               Multi-Agent Systems and Negotiation|      Artificial Intelligence| 11|            4.0|
|Intelligent Tutoring Systems and Adaptive Learning|      Artificial Intelligence| 55|           3.32|
|                  Nanofluid Flow and Heat Transfer|       Biomedical Engineering| 11|           2.55|
|                   Language and cultural evolution|             Cultural Studies| 11|           2.38|
|                 Translation Studies and Practices|     Language and Linguistics| 12|           2.29|
|               E-Learning and Knowledge Management|Computer Science Applications| 28|           2.12

**Lectura (corte 2026-10-09).** El 8,7 % de los tópico-años etiquetables resultan emergentes (289 de 3 308). En
el último año base etiquetado, 2023, la tasa es 9,1 % (45 de 494). Entre los emergentes de 2023 hay sistemas
multiagente y negociación, tutores inteligentes y aprendizaje adaptativo, y web semántica y ontologías.

### 8.5 Subcampos

**Qué hace.** Mide la participación de cada subcampo en las publicaciones del año al inicio y al final del
último horizonte completo de 2 años (2023 y 2025), el mismo horizonte que usa la etiqueta.

**Por qué participación y no conteos.** Entre 2023 y 2025 las publicaciones totales de la muestra se duplican
(×2,1). Con conteos, casi todos los subcampos parecen crecer; con participación, solo crecen los que ganan
espacio frente al resto.

**Muestra.** Dos tablas con las mismas columnas: los 10 subcampos con más publicaciones en 2025, y los 10 que
más participación ganan.
- `share_start_pct` y `share_end_pct`: participación en 2023 y 2025, en %.
- `change_pp`: diferencia en puntos porcentuales.
- `relative_change_pct`: cambio relativo de la participación, en %.

In [13]:
# 8.5 Subfield share of the year's publications at the start and end of the last complete two-year horizon.
start_year, end_year = LAST_COMPLETE_YEAR - HORIZON_YEARS, LAST_COMPLETE_YEAR
subfield_share = (
    topic_year.filter(F.col("year").isin(start_year, end_year))
    .groupBy("subfield", "year")
    .agg(F.sum("n_t").alias("n"), F.first("n_total").alias("n_total"))
    .withColumn("share", F.col("n") / F.col("n_total"))
    .groupBy("subfield")
    .agg(
        F.sum(F.when(F.col("year") == end_year, F.col("n"))).alias(f"n_{end_year}"),
        F.sum(F.when(F.col("year") == start_year, F.col("share"))).alias("share_start"),
        F.sum(F.when(F.col("year") == end_year, F.col("share"))).alias("share_end"),
    )
    .select(
        "subfield",
        f"n_{end_year}",
        F.round(F.col("share_start") * 100, 2).alias("share_start_pct"),
        F.round(F.col("share_end") * 100, 2).alias("share_end_pct"),
        F.round((F.col("share_end") - F.col("share_start")) * 100, 2).alias("change_pp"),
        F.round((F.col("share_end") / F.col("share_start") - 1) * 100, 1).alias("relative_change_pct"),
    )
)
print(f"largest subfields in {end_year} (share {start_year} -> {end_year})")
subfield_share.orderBy(F.desc(f"n_{end_year}"), "subfield").show(10, truncate=45)
print(f"largest share gains {start_year} -> {end_year}")
subfield_share.orderBy(F.desc("change_pp"), "subfield").show(10, truncate=45)

largest subfields in 2025 (share 2023 -> 2025)


+------------------------------------------+------+---------------+-------------+---------+-------------------+
|                                  subfield|n_2025|share_start_pct|share_end_pct|change_pp|relative_change_pct|
+------------------------------------------+------+---------------+-------------+---------+-------------------+
|                   Artificial Intelligence| 14589|          18.65|        20.05|      1.4|                7.5|
|                        Health Informatics|  5904|           7.59|         8.12|     0.53|                7.0|
|                       Information Systems|  5480|            5.0|         7.53|     2.54|               50.8|
|   Computer Vision and Pattern Recognition|  2550|           5.29|         3.51|    -1.78|              -33.7|
|                           Safety Research|  2117|           2.28|         2.91|     0.63|               27.5|
|      Computer Networks and Communications|  1808|           2.73|         2.49|    -0.24|             

+------------------------------------------+------+---------------+-------------+---------+-------------------+
|                                  subfield|n_2025|share_start_pct|share_end_pct|change_pp|relative_change_pct|
+------------------------------------------+------+---------------+-------------+---------+-------------------+
|                       Information Systems|  5480|            5.0|         7.53|     2.54|               50.8|
|                   Artificial Intelligence| 14589|          18.65|        20.05|      1.4|                7.5|
|             Computer Science Applications|  1597|            1.3|          2.2|      0.9|               69.2|
|                           Safety Research|  2117|           2.28|         2.91|     0.63|               27.5|
|                        Health Informatics|  5904|           7.59|         8.12|     0.53|                7.0|
|                                 Education|   686|           0.43|         0.94|     0.51|             

**Lectura (corte 2026-10-09).** Inteligencia artificial sigue siendo el subcampo más grande y gana 1,4 puntos
(18,65 % a 20,05 %). Las mayores ganancias relativas entre los subcampos grandes son Information Systems (+2,5
puntos, +51 %) y Computer Science Applications (+69 %). Computer Vision and Pattern Recognition pierde
participación (5,29 % a 3,51 %, −34 %) aunque sus publicaciones aumentan: crece menos que el resto del campo.

### 8.6 Señales actuales (sin etiqueta)

**Qué hace.** Busca, en el último año completo (2025), tópicos con soporte suficiente (al menos 10
publicaciones, el mismo mínimo de la etiqueta) y los ordena por el crecimiento de su prevalencia en los dos
años anteriores.

**Muestra.** Tópico, subcampo, publicaciones (`n_t`), prevalencia, crecimiento a 1 año (`growth_1y`),
crecimiento a 2 años (`growth_2y_past`; 2,0 = la prevalencia se triplicó) y años seguidos de crecimiento.

**Cómo leerlo.** Estas filas **no tienen etiqueta**: su horizonte termina después del último año completo. Son
candidatos, no tópicos emergentes confirmados; el modelo es el que debe estimar su probabilidad. Con soportes
cercanos a 10 publicaciones, un crecimiento alto puede deberse a pocas publicaciones.

In [14]:
# 8.6 Current signals: last complete year, enough support, ranked by past two-year prevalence growth. Unlabelled rows.
current = topic_year.filter(
    (F.col("year") == LAST_COMPLETE_YEAR) & (F.col("n_t") >= MIN_SUPPORT) & F.col("growth_2y_past").isNotNull()
)
print(f"topics with at least {MIN_SUPPORT} publications in {LAST_COMPLETE_YEAR}: {current.count()}")
(
    current.orderBy(F.desc("growth_2y_past"), "topic_id")
    .select(
        "topic_name", "subfield", "n_t",
        F.round("prevalence_t", 6).alias("prevalence_t"),
        F.round("growth_1y", 2).alias("growth_1y"),
        F.round("growth_2y_past", 2).alias("growth_2y_past"),
        "consecutive_growth_years",
    )
    .show(15, truncate=45)
)

topics with at least 10 publications in 2025: 802


+---------------------------------------------+------------------------------------------+---+------------+---------+--------------+------------------------+
|                                   topic_name|                                  subfield|n_t|prevalence_t|growth_1y|growth_2y_past|consecutive_growth_years|
+---------------------------------------------+------------------------------------------+---+------------+---------+--------------+------------------------+
|            Personality Traits and Psychology|                       Clinical Psychology| 29|     3.99E-4|     0.15|         12.64|                       2|
|               Expert finding and Q&A systems|                       Information Systems| 20|     2.75E-4|    -0.02|          8.41|                       0|
|     Library Science and Information Literacy|          Library and Information Sciences| 18|     2.47E-4|     2.82|          7.47|                       3|
|       Security and Verification in Computing|     

### 8.7 Resumen de KPI

**Qué hace.** Reúne en una tabla los KPI principales de las secciones 8.1 a 8.4, con el periodo de cada uno.
Es la tabla para reportar resultados; cada valor sale de una celda anterior de este notebook.

In [15]:
# 8.7 KPI summary: one row per headline KPI with its period and value, all taken from the cells above.
first_growth, last_growth = growth_by_year[first_year], growth_by_year[last_year]
first_concentration, last_concentration = concentration_by_year[first_year], concentration_by_year[last_year]
first_profile, last_profile = profile_by_year[first_year], profile_by_year[last_year]
span = f"{first_year} -> {last_year}"
kpi_rows = [
    ("Publication compound annual growth", f"{first_year}-{last_year}", f"{cagr:.1%}"),
    ("Publication year-over-year change", str(last_year), f"{last_growth['yoy_growth']:+.1%}"),
    ("Active topics", span, f"{first_growth['active_topics']} -> {last_growth['active_topics']}"),
    ("Share of the 10 largest topics", span, f"{first_concentration['top10_share']:.1%} -> {last_concentration['top10_share']:.1%}"),
    ("Effective topics (1 / HHI)", span, f"{1 / first_concentration['hhi']:.0f} -> {1 / last_concentration['hhi']:.0f}"),
    ("Open access share", span, f"{first_profile['open_access']:.1%} -> {last_profile['open_access']:.1%}"),
    ("Conference share", span, f"{first_profile['conference']:.1%} -> {last_profile['conference']:.1%}"),
    ("Authors per publication", span, f"{first_profile['authors_per_pub']:.2f} -> {last_profile['authors_per_pub']:.2f}"),
    ("Emergence rate (all labelled base years)", f"{first_year}-{last_labelled_year}",
     f"{overall['positives'] / overall['labelable']:.1%} ({overall['positives']} / {overall['labelable']})"),
    ("Emergence rate (last labelled base year)", str(last_labelled_year),
     f"{latest['positives'] / latest['labelable']:.1%} ({latest['positives']} / {latest['labelable']})"),
]
spark.createDataFrame(kpi_rows, "kpi string, period string, value string").show(truncate=False)

+----------------------------------------+------------+-----------------+
|kpi                                     |period      |value            |
+----------------------------------------+------------+-----------------+
|Publication compound annual growth      |2011-2025   |20.9%            |
|Publication year-over-year change       |2025        |+57.2%           |
|Active topics                           |2011 -> 2025|976 -> 2727      |
|Share of the 10 largest topics          |2011 -> 2025|23.4% -> 26.3%   |
|Effective topics (1 / HHI)              |2011 -> 2025|80 -> 77         |
|Open access share                       |2011 -> 2025|22.9% -> 64.1%   |
|Conference share                        |2011 -> 2025|30.3% -> 25.5%   |
|Authors per publication                 |2011 -> 2025|2.85 -> 4.07     |
|Emergence rate (all labelled base years)|2011-2023   |8.7% (289 / 3308)|
|Emergence rate (last labelled base year)|2023        |9.1% (45 / 494)  |
+-------------------------------------

## Cierre

Libera las tablas en memoria y detiene la sesión de Spark.

In [16]:
# Release cached tables and stop Spark.
topic_year.unpersist()
labelable.unpersist()
spark.stop()